# Notebook 07 — Encoder-Decoder Transformer for Translation

## Learning Objectives
- Understand the encoder-decoder architecture for sequence-to-sequence tasks.
- Load EN→FR translation pairs from CSV using `src.data.synthetic_translation`.
- Train a `Seq2SeqTransformer` with teacher forcing.
- Use greedy decoding to generate translations at inference time.
- Visualize the cross-attention map from decoder to encoder.

## Big Picture
Machine translation requires *reading* the entire source sentence and *generating*
the target sentence word by word. The encoder builds a rich representation of the source;
the decoder attends to it via **cross-attention** while generating the target one token at a time.

## 1. Imports and Setup

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from src.utils.seed import seed_everything
from src.utils.device import get_best_device
from src.utils.cleanup import prepare_run_dir
from src.utils.reporting import save_metrics_json, save_markdown_report, save_table_csv
from src.utils.paths import RUNS_TRANSLATION_DIR, TINY_TRANSLATION_CSV
from src.data.synthetic_translation import (
    load_translation_data, BOS_IDX, EOS_IDX, PAD_IDX,
    encode_source, encode_target_input, tokenize,
)
from src.models.seq2seq_transformer import Seq2SeqTransformer, greedy_decode

seed_everything(42)
device = get_best_device()
prepare_run_dir(RUNS_TRANSLATION_DIR, clean=True)
print(f"Device  : {device}")
print(f"Run dir : {RUNS_TRANSLATION_DIR}")

[seed] Random seed set to 42.
[cleanup] Removed 5 old file(s) from runs/translation
Device  : cuda
Run dir : /home/arash/PycharmProjects/Transformer/transformers-from-scratch/runs/translation


## 2. Dataset: Tiny English-to-French Translation Pairs

In [2]:
# Inspect raw CSV
df = pd.read_csv(TINY_TRANSLATION_CSV)
print(f"Dataset path  : {TINY_TRANSLATION_CSV}")
print(f"Total pairs   : {len(df)}")
print(f"Columns       : {list(df.columns)}")
print(f"\nFirst 5 pairs:")
print(df.head())

Dataset path  : /home/arash/PycharmProjects/Transformer/transformers-from-scratch/data/tiny/tiny_translation_en_fr.csv
Total pairs   : 900
Columns       : ['source', 'target']

First 5 pairs:
          source             target
0     I am happy    je suis heureux
1       I am sad     je suis triste
2  You are tired      tu es fatigue
3     He is tall       il est grand
4    She is kind  elle est gentille


In [3]:
# Hyperparameters
MAX_SRC_LEN = 12
MAX_TGT_LEN = 12
BATCH_SIZE  = 16
D_MODEL     = 96
NUM_HEADS   = 4
N_ENC       = 2
N_DEC       = 2
DIM_FF      = 256
NUM_EPOCHS  = 100
LR          = 3e-4

# Load data
train_loader, val_loader, src_vocab, tgt_vocab = load_translation_data(
    csv_path=TINY_TRANSLATION_CSV,
    max_src_len=MAX_SRC_LEN,
    max_tgt_len=MAX_TGT_LEN,
    batch_size=BATCH_SIZE,
    num_workers=0,
    seed=42,
)

SRC_VOCAB_SIZE = len(src_vocab)
TGT_VOCAB_SIZE = len(tgt_vocab)
tgt_idx2word   = {v: k for k, v in tgt_vocab.items()}
src_idx2word   = {v: k for k, v in src_vocab.items()}

print(f"Source vocab size : {SRC_VOCAB_SIZE}")
print(f"Target vocab size : {TGT_VOCAB_SIZE}")
print(f"Train batches     : {len(train_loader)}")
print(f"Val   batches     : {len(val_loader)}")

# Inspect one batch
src_batch, tgt_in_batch, tgt_out_batch = next(iter(train_loader))
print(f"\nsrc_ids shape    : {src_batch.shape}     (batch, max_src_len)")
print(f"tgt_input shape  : {tgt_in_batch.shape}   (batch, max_tgt_len) — starts with BOS")
print(f"tgt_output shape : {tgt_out_batch.shape}   (batch, max_tgt_len) — ends with EOS")

Source vocab size : 240
Target vocab size : 302
Train batches     : 48
Val   batches     : 9

src_ids shape    : torch.Size([16, 12])     (batch, max_src_len)
tgt_input shape  : torch.Size([16, 12])   (batch, max_tgt_len) — starts with BOS
tgt_output shape : torch.Size([16, 12])   (batch, max_tgt_len) — ends with EOS


## 3. Architecture (Text Diagram)

```
Source: "the cat is on the mat"
Target: "le chat est sur le tapis"

ENCODER SIDE
  src_ids → src_embedding × √d_model + SinPE
  → TransformerEncoder (2 blocks)
  → encoder_output [batch, src_len, d_model]

DECODER SIDE (teacher forcing during training)
  tgt_ids_in (BOS + target) → tgt_embedding × √d_model + SinPE
  → TransformerDecoder (2 blocks)
      Block i:
        1. Masked Self-Attention     (causal — can't see future target tokens)
        2. Cross-Attention           (queries from decoder, keys/values from encoder)
        3. FeedForward
  → output projection → logits [batch, tgt_len, tgt_vocab_size]
  → Cross-Entropy vs tgt_ids_out (target + EOS)

INFERENCE (greedy decoding)
  Encode source once → generate target token by token until EOS



SOURCE SIDE
────────────────────────────────────────────

src_ids [B, 12]
    │
    ▼
src_embedding × √d_model + PE
[B, 12, 96]
    │
    ▼
┌────────────────────────────────────────────┐
│ Encoder Block × 2                          │
│                                            │
│ Self-Attention: source → source            │
│                                            │
│ Q = Wq(source)  [B, 4, 12, 24]             │
│ K = Wk(source)  [B, 4, 12, 24]             │
│ V = Wv(source)  [B, 4, 12, 24]             │
│                                            │
│ softmax(QKᵀ / √24) V                       │
│ → [B, 4, 12, 24]                           │
│ concat heads → [B, 12, 96]                 │
│                                            │
│ FeedForward: 96 → 256 → 96                 │
└────────────────────────────────────────────┘
    │
    ▼
encoder_output / memory
[B, 12, 96]


TARGET SIDE
────────────────────────────────────────────

tgt_input [B, 12]
    │
    ▼
tgt_embedding × √d_model + PE
[B, 12, 96]
    │
    ▼
┌────────────────────────────────────────────┐
│ Decoder Block × 2                          │
│                                            │
│ 1) Masked Self-Attention: target → target  │
│                                            │
│ Q = Wq(target) [B, 4, 12, 24]              │
│ K = Wk(target) [B, 4, 12, 24]              │
│ V = Wv(target) [B, 4, 12, 24]              │
│                                            │
│ causal mask applied                        │
│ softmax(QKᵀ / √24) V                       │
│ → [B, 12, 96]                              │
│                                            │
│ 2) Cross-Attention: target → source        │
│                                            │
│ Q = Wq(decoder_state) [B, 4, 12, 24]       │
│ K = Wk(encoder_output) [B, 4, 12, 24]      │
│ V = Wv(encoder_output) [B, 4, 12, 24]      │
│                                            │
│ softmax(QKᵀ / √24) V                       │
│ cross_weights [B, 4, tgt_len, src_len]     │
│ → [B, 12, 96]                              │
│                                            │
│ 3) FeedForward: 96 → 256 → 96              │
└────────────────────────────────────────────┘
    │
    ▼
decoder_output
[B, 12, 96]
    │
    ▼
Linear projection
96 → tgt_vocab_size
    │
    ▼
logits
[B, 12, TGT_VOCAB_SIZE]
```

## 4. Theory: Teacher Forcing and Greedy Decoding

**Teacher forcing** (training): Feed the ground-truth target tokens as decoder input.
At step t, decoder input = [BOS, y_1, y_2, ..., y_{t-1}].
The causal mask prevents the model from cheating by looking at y_t when predicting y_t.

**Greedy decoding** (inference): No ground truth available.
At step t, feed the model's own prediction from step t-1 as input.
Pick the highest-probability token at each step: ŷ_t = argmax softmax(logits_t).
Stop when EOS is generated.

**Cross-attention**: Decoder queries (position-in-target) attend to encoder keys/values (source tokens).
This is the bridge that lets the decoder know what source tokens to translate at each step.

## 5. Implementation: Build and Inspect the Model

In [4]:
model = Seq2SeqTransformer(
    src_vocab_size=SRC_VOCAB_SIZE,
    tgt_vocab_size=TGT_VOCAB_SIZE,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    num_encoder_layers=N_ENC,
    num_decoder_layers=N_DEC,
    dim_feedforward=DIM_FF,
    max_src_len=MAX_SRC_LEN + 4,
    max_tgt_len=MAX_TGT_LEN + 4,
    dropout=0.05,
    pad_idx=PAD_IDX,
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")

# Forward pass shape trace
src = src_batch[:4].to(device)
tgt_in = tgt_in_batch[:4].to(device)

model.eval()
with torch.no_grad():
    logits, cross_w = model(src, tgt_in, return_cross_weights=True)

print(f"\nsrc shape           : {src.shape}        (batch, src_len)")
print(f"tgt_in shape        : {tgt_in.shape}     (batch, tgt_len)")
print(f"logits shape        : {logits.shape}  (batch, tgt_len, tgt_vocab_size)")
if cross_w is not None:
    print(f"cross_weights shape : {cross_w.shape}   (batch, heads, tgt_len, src_len)")

Trainable parameters: 502,830

src shape           : torch.Size([4, 12])        (batch, src_len)
tgt_in shape        : torch.Size([4, 12])     (batch, tgt_len)
logits shape        : torch.Size([4, 12, 302])  (batch, tgt_len, tgt_vocab_size)
cross_weights shape : torch.Size([4, 4, 12, 12])   (batch, heads, tgt_len, src_len)


## 6. Sanity Checks

In [5]:
# Shape checks
assert logits.shape == (4, MAX_TGT_LEN, TGT_VOCAB_SIZE), f"{logits.shape}"
print("Logits shape check passed.")

if cross_w is not None:
    assert cross_w.shape == (4, NUM_HEADS, MAX_TGT_LEN, MAX_SRC_LEN), f"{cross_w.shape}"
    # Cross-attention weights should sum to 1 over source dimension
    row_sums = cross_w.sum(dim=-1)
    assert torch.allclose(row_sums, torch.ones_like(row_sums), atol=1e-5)
    print("Cross-attention weights sum-to-1 check passed.")

# Initial loss should be close to log(tgt_vocab_size)
import math
B, T, V = logits.shape
init_loss = F.cross_entropy(
    logits.reshape(B*T, V),
    tgt_out_batch[:4].to(device).reshape(B*T),
    ignore_index=PAD_IDX,
).item()
expected_loss = math.log(TGT_VOCAB_SIZE)
print(f"Initial loss: {init_loss:.4f}  (expected ≈ {expected_loss:.4f} = log({TGT_VOCAB_SIZE}))")
print("All sanity checks passed!")

Logits shape check passed.
Cross-attention weights sum-to-1 check passed.
Initial loss: 5.9604  (expected ≈ 5.7104 = log(302))
All sanity checks passed!


## 7. Training

In [6]:
from src.training.translation import translation_loss_fn, translation_token_accuracy

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    betas=(0.9, 0.98),
    eps=1e-9,
    weight_decay=1e-4,
)

train_losses, val_losses = [], []
train_accs,   val_accs   = [], []

best_val_loss = float("inf")
best_val_acc = 0.0
best_epoch = 0
best_state = None

for epoch in range(1, NUM_EPOCHS + 1):
    # ── Train ──
    model.train()
    ep_loss, n_batches = 0.0, 0

    for batch in train_loader:
        optimizer.zero_grad()
        loss = translation_loss_fn(model, batch, device)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        ep_loss += loss.item()
        n_batches += 1

    train_losses.append(ep_loss / max(n_batches, 1))

    # ── Train accuracy ──
    model.eval()
    train_tok_acc = []

    with torch.no_grad():
        for batch in train_loader:
            metrics = translation_token_accuracy(model, batch, device)
            train_tok_acc.append(metrics["token_accuracy"])

    train_accs.append(sum(train_tok_acc) / max(len(train_tok_acc), 1))

    # ── Validate ──
    v_loss, v_batches = 0.0, 0
    val_tok_acc = []

    with torch.no_grad():
        for batch in val_loader:
            v_loss += translation_loss_fn(model, batch, device).item()
            metrics = translation_token_accuracy(model, batch, device)
            val_tok_acc.append(metrics["token_accuracy"])
            v_batches += 1

    current_val_loss = v_loss / max(v_batches, 1)
    current_val_acc = sum(val_tok_acc) / max(len(val_tok_acc), 1)

    val_losses.append(current_val_loss)
    val_accs.append(current_val_acc)

    # Save best model by validation loss
    if current_val_loss < best_val_loss:
        best_val_loss = current_val_loss
        best_val_acc = current_val_acc
        best_epoch = epoch
        best_state = {
            k: v.detach().cpu().clone()
            for k, v in model.state_dict().items()
        }

    if epoch % 10 == 0 or epoch == 1:
        print(
            f"Epoch {epoch:3d}/{NUM_EPOCHS}  "
            f"train_loss={train_losses[-1]:.4f}  train_tok_acc={train_accs[-1]:.3f}  "
            f"val_loss={val_losses[-1]:.4f}  val_tok_acc={val_accs[-1]:.3f}"
        )

# Restore best model before decoding/evaluation
if best_state is not None:
    model.load_state_dict({
        k: v.to(device)
        for k, v in best_state.items()
    })

print(
    f"Training complete! "
    f"Best epoch: {best_epoch} | "
    f"best_val_loss={best_val_loss:.4f} | "
    f"best_val_tok_acc={best_val_acc:.3f}"
)

Epoch   1/100  train_loss=4.1380  train_tok_acc=0.478  val_loss=3.0619  val_tok_acc=0.454
Epoch  10/100  train_loss=0.4633  train_tok_acc=0.942  val_loss=0.6958  val_tok_acc=0.885
Epoch  20/100  train_loss=0.1078  train_tok_acc=0.994  val_loss=0.5262  val_tok_acc=0.917
Epoch  30/100  train_loss=0.0263  train_tok_acc=0.998  val_loss=0.5372  val_tok_acc=0.913
Epoch  40/100  train_loss=0.0093  train_tok_acc=0.999  val_loss=0.5862  val_tok_acc=0.915
Epoch  50/100  train_loss=0.0081  train_tok_acc=0.999  val_loss=0.6327  val_tok_acc=0.915
Epoch  60/100  train_loss=0.0049  train_tok_acc=0.999  val_loss=0.6445  val_tok_acc=0.914
Epoch  70/100  train_loss=0.0060  train_tok_acc=0.999  val_loss=0.6598  val_tok_acc=0.922
Epoch  80/100  train_loss=0.0031  train_tok_acc=0.999  val_loss=0.7010  val_tok_acc=0.922
Epoch  90/100  train_loss=0.0026  train_tok_acc=0.999  val_loss=0.7476  val_tok_acc=0.915
Epoch 100/100  train_loss=0.0027  train_tok_acc=0.999  val_loss=0.7738  val_tok_acc=0.916
Training c

## 8. Evaluation: Translation Examples

In [7]:
# Greedy decode on validation set samples
model.eval()

def decode_ids(ids_tensor, idx2word):
    """Convert tensor of token IDs to a readable string."""
    tokens = []
    for idx in ids_tensor.tolist():
        if idx == EOS_IDX:
            break
        if idx not in (PAD_IDX, BOS_IDX):
            tokens.append(idx2word.get(idx, '<unk>'))
    return ' '.join(tokens)

print("Translation examples (greedy decoding):")
print("=" * 72)

translation_rows = []
for batch in val_loader:
    src_ids, tgt_in_ids, tgt_out_ids = batch
    for i in range(min(5, src_ids.size(0))):
        src_single = src_ids[i].unsqueeze(0).to(device)  # [1, src_len]
        generated  = greedy_decode(
            model, src_single, BOS_IDX, EOS_IDX,
            max_len=MAX_TGT_LEN, device=device
        )
        src_str  = decode_ids(src_ids[i],     src_idx2word)
        ref_str  = decode_ids(tgt_out_ids[i], tgt_idx2word)
        pred_str = decode_ids(generated,       tgt_idx2word)

        print(f"  Source : {src_str}")
        print(f"  Target : {ref_str}")
        print(f"  Pred   : {pred_str}")
        print()
        translation_rows.append({'source': src_str, 'reference': ref_str, 'prediction': pred_str})
    break

# Save translation examples to CSV
tbl_path = RUNS_TRANSLATION_DIR / 'translation_examples.csv'
save_table_csv(translation_rows, tbl_path)
print(f"Translation table saved to: {tbl_path}")

Translation examples (greedy decoding):
  Source : the angry tree is in the garden
  Target : l arbre en colere est dans le jardin
  Pred   : l arbre en colere est dans le jardin

  Source : the happy garden is in the house
  Target : le jardin heureux est dans la maison
  Pred   : le jardin heureux est dans la maison

  Source : he likes this song
  Target : il aime cette chanson
  Pred   : il aime sa heureux

  Source : the tired dog is in the garden
  Target : le chien fatigue est dans le jardin
  Pred   : le chien fatigue est dans le jardin

  Source : the flower is clear
  Target : la fleur est claire
  Pred   : la fleur est claire

Translation table saved to: /home/arash/PycharmProjects/Transformer/transformers-from-scratch/runs/translation/translation_examples.csv


## 9. Visualization

In [8]:
# Figure 1: Training and validation loss curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ep = range(1, NUM_EPOCHS + 1)

ax1.plot(ep, train_losses, 'o-', markersize=3, label='Train', color='steelblue')
ax1.plot(ep, val_losses,   's-', markersize=3, label='Val',   color='tomato')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Cross-Entropy Loss')
ax1.set_title('Translation Loss'); ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(ep, [a*100 for a in train_accs], 'o-', markersize=3, label='Train', color='steelblue')
ax2.plot(ep, [a*100 for a in val_accs],   's-', markersize=3, label='Val',   color='tomato')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Token Accuracy (%)')
ax2.set_title('Token Accuracy'); ax2.legend(); ax2.grid(True, alpha=0.3)

fig.suptitle('Seq2Seq Transformer: EN→FR Translation', fontsize=12)
fig.tight_layout()
curve_path = RUNS_TRANSLATION_DIR / 'loss_curve.png'
fig.savefig(curve_path, dpi=120)
plt.close(fig)
print(f"Loss curve saved to: {curve_path}")

Loss curve saved to: /home/arash/PycharmProjects/Transformer/transformers-from-scratch/runs/translation/loss_curve.png


In [9]:
# Figure 2: Cross-attention heatmap for a single example
model.eval()
for batch in val_loader:
    src_ids, tgt_in_ids, _ = batch
    src_single   = src_ids[0:1].to(device)
    tgt_in_single = tgt_in_ids[0:1].to(device)

    with torch.no_grad():
        logits, cross_w = model(src_single, tgt_in_single, return_cross_weights=True)

    if cross_w is not None:
        # cross_w: [1, heads, tgt_len, src_len] — average over heads
        avg_cross_w = cross_w[0].mean(dim=0).cpu().numpy()  # [tgt_len, src_len]

        src_tokens = [src_idx2word.get(int(t), '<pad>') for t in src_ids[0] if int(t) != PAD_IDX]
        tgt_tokens = [tgt_idx2word.get(int(t), '<pad>') for t in tgt_in_ids[0]
                      if int(t) not in (PAD_IDX,)]

        n_src = len(src_tokens)
        n_tgt = len(tgt_tokens)

        fig, ax = plt.subplots(figsize=(max(6, n_src), max(5, n_tgt // 2 + 2)))
        im = ax.imshow(avg_cross_w[:n_tgt, :n_src], cmap='Blues', aspect='auto')
        ax.set_xticks(range(n_src))
        ax.set_yticks(range(n_tgt))
        ax.set_xticklabels(src_tokens, rotation=45, ha='right')
        ax.set_yticklabels(tgt_tokens)
        ax.set_xlabel('Source (English)')
        ax.set_ylabel('Target (French)')
        ax.set_title('Cross-Attention: Decoder → Encoder\n(averaged over heads)')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        fig.tight_layout()

        ca_path = RUNS_TRANSLATION_DIR / 'cross_attention.png'
        fig.savefig(ca_path, dpi=120)
        plt.close(fig)
        print(f"Cross-attention heatmap saved to: {ca_path}")
    break

Cross-attention heatmap saved to: /home/arash/PycharmProjects/Transformer/transformers-from-scratch/runs/translation/cross_attention.png


In [10]:
# Save session report
metrics = {
    'best_epoch': best_epoch,
    'best_val_loss': best_val_loss,
    'best_val_token_acc': best_val_acc,
    'final_val_loss_last_epoch': val_losses[-1],
    'final_val_token_acc_last_epoch': val_accs[-1],
    'final_train_loss': train_losses[-1],
    'num_epochs': NUM_EPOCHS,
    'src_vocab_size': SRC_VOCAB_SIZE,
    'tgt_vocab_size': TGT_VOCAB_SIZE,
    'num_params': n_params,
}

save_metrics_json(metrics, RUNS_TRANSLATION_DIR / 'metrics.json')

save_markdown_report(
    title='EN→FR Translation',
    summary=(
        f'Seq2SeqTransformer trained for {NUM_EPOCHS} epochs. '
        f'Best epoch: {best_epoch}. '
        f'Best val loss: {best_val_loss:.4f}, '
        f'best token acc: {best_val_acc:.4f}.'
    ),
    metrics=metrics,
    figures=['loss_curve.png', 'cross_attention.png'],
    tables=['translation_examples.csv'],
    output_path=RUNS_TRANSLATION_DIR / 'session_report.md',
    device=str(device),
    hyperparams={
        'd_model': D_MODEL,
        'heads': NUM_HEADS,
        'enc_layers': N_ENC,
        'dec_layers': N_DEC,
        'dim_feedforward': DIM_FF,
        'lr': LR,
        'batch_size': BATCH_SIZE,
        'max_src_len': MAX_SRC_LEN,
        'max_tgt_len': MAX_TGT_LEN,
    },
    architecture=(
        f'Seq2SeqTransformer: src_emb(d_model={D_MODEL}) + PE + '
        f'Encoder × {N_ENC} | tgt_emb(d_model={D_MODEL}) + PE + '
        f'Decoder × {N_DEC} + projection({D_MODEL}→{TGT_VOCAB_SIZE})'
    ),
    loss_fn='CrossEntropyLoss (ignore_index=PAD)',
)

print(f"Session report saved to: {RUNS_TRANSLATION_DIR / 'session_report.md'}")

Session report saved to: /home/arash/PycharmProjects/Transformer/transformers-from-scratch/runs/translation/session_report.md


## 10. Failure Cases

**Model outputs repetitions**: With greedy decoding, the model sometimes loops, outputting the same token repeatedly. Solutions: beam search, top-k sampling, or nucleus sampling.

**Causal mask missing**: Without the causal mask in the decoder's self-attention, the model cheats by looking at future target tokens during training but fails during inference (no future tokens available).

**Teacher forcing exposure bias**: The model never sees its own mistakes during training. At inference, errors compound — each wrong token leads to a worse next prediction. Fix: scheduled sampling.

**Small vocab + small data**: With 900 sentence pairs, most translations are simple lookups. The model may memorize without truly learning translation.

**Overfitting after best validation epoch**: In this run, training token accuracy reaches 100%, while validation loss improves only up to the best checkpoint and then starts to rise. For this reason, decoding and reporting use the best validation-loss checkpoint instead of the last epoch.

## 11. Exercises

1. **Beam search**: Implement a simple beam search decoder with beam_size=3. Does it produce better translations than greedy?
2. **Cross-attention alignment**: For a translation you understand (e.g., "the cat" → "le chat"), check that the cross-attention map aligns the corresponding source and target words.
3. **Encoder ablation**: Reduce encoder layers from 2 to 1. Does translation quality drop? How about increasing to 4 layers?
4. **Vocabulary size effect**: The target vocab is small. What happens if you lower `vocab_min_freq` to exclude rare words? How does OOV rate change?
5. **BLEU score**: Implement a simple BLEU-1 (unigram precision with brevity penalty). Calculate it on the validation set.

## 12. Key Takeaways

- The **encoder-decoder Transformer** uses the encoder for reading, the decoder for generating.
- **Cross-attention** bridges encoder and decoder: decoder queries attend to encoder keys/values.
- **Teacher forcing** (training) feeds ground-truth tokens to the decoder; **greedy decoding** (inference) feeds the model's own predictions.
- The **causal mask** in the decoder prevents future-token leakage during training.
- Cross-attention heatmaps reveal word alignment — a natural byproduct of the attention mechanism.